# Usage Examples

Two end-to-end examples of driving the team with `run_research`, plus a monitor-inspection helper. The runtime agents see only `prompts/` content; this notebook is for human inspection after a run.

1. **Using existing tools** — a question the current `xspec_utils` toolkit can already answer (a joint broadband spectral fit).
2. **Test code authoring** — a question that needs capability the toolkit does *not* expose, so the analyst lead authors the missing tools into `research_toolkit` and the workers rerun on them (Phase 2 step 7).

In [1]:
from datetime import datetime
import logging
from pathlib import Path
import sys
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv(override=True)

for noisy_logger in ("arxiv", "httpcore", "httpx"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

from auto_research import run_research
from auto_research.monitor import RunMonitor
from auto_research.schemas import ResearchRequest

## Using existing tools

`chatter=7` prints compact live monitor lines while the run is active. The JSONL log still receives every event, including level-10 model-call details. This question is answerable with the toolkit as it stands (`load_data`, `fit_tbfeo_hss`, `run_fit`, `xspec_status`, `plot_fit`, `plot_iqu`, `group_pha`).

In [ ]:
request = ResearchRequest(
    question=(
            "IXPE has performed a Cyg X-1 observation in its soft state on 2023-05-24. "
            "Find simultaneous or near-simultaneous observations from other X-ray observatories "
            "(e.g. NICER, NuSTAR, XMM-Newton) and perform a joint broadband spectral fit "
            "for this epoch. epoch including observations: IXPE: '02008401', NICER: '6643010101' and '6643010102', NuSTAR: '80902318002'."
        ),
        context=(
            "TEST SESSION — evaluation criteria are relaxed as follows:\n"
            "- Absent NuSTAR and NICER backgrounds are acceptable and expected: "
            "Cyg X-1 is an extremely bright source for which background files are not used "
            "(background contribution is negligible; this is standard practice for bright sources).\n"
            "- The primary goal is to verify that analysts correctly invoked the available "
            "toolkit tools (load_data, fit_tbfeo_hss, run_fit, xspec_status, plot_fit, "
            "plot_iqu, group_pha) and that any inline Python executed via "
            "`mcp.builtin.bash_exec` with `python -c` successfully imports xspec and "
            "xspec_utils without errors.\n"
            "- The existing toolkit will not produce publication-level results. "
            "You may send back one round of revision to analysts, asking for better results. "
            "They should attempt to run inline calculations via python -c until they get "
            "non-error outputs and reasonable-looking fit statistics, "
            "but they are not expected to achieve publication-quality fits.\n"
            "- Accept the analysis if tools were called with correct arguments and returned "
            "non-error outputs, even if fit statistics are not publication-quality.\n"
            "- If any of the tools or environments seemed broken, abort the run immediately and report."
        )
)

LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/P1/step-12")
LOG_DIR.mkdir(parents=True, exist_ok=True)
monitor_jsonl = LOG_DIR / f"cyg_x1_ixpe_softstate_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.jsonl"
monitor = RunMonitor(chatter=7, jsonl_path=monitor_jsonl, stream=sys.stdout)
run = await run_research(request, monitor=monitor)
run.monitor.write_markdown(monitor_jsonl.with_suffix(".md"))
monitor_jsonl, monitor_jsonl.with_suffix(".md"), run.verdict

In [ ]:
display(Markdown(run.markdown()))

## Test code authoring

This poses IXPE project **P1 — intra-observation PA/PD stability** (`bhxrb_intraobservation_pa_pd_stability`). Every measurement routes through time-resolved `xpbin algorithm=PCUBE` (PD, PA, Q/I, U/I, MDP99 in ~5 ks segments). The current `xspec_utils` `TOOL_MANIFEST` exposes **only** the spectral-fit workflow — it does not surface PCUBE, the IXPE reduction scripts, or any HEASARC acquire path — so a worker hits the gap and raises a `tool_request`. The analyst lead then authors the missing capability through the gated flow (author → gate → GPT co-lead review → atomic commit on branch `ai`), the registry reloads, sandboxes restart, and the workers rerun on the new tool. With data not local, the realistic chain is **heasarc acquire (new package) → reduction wiring → pcube exposure**, bounded by `tool_author_cap`.

**Prerequisites (human-staged, out of band):** `research_toolkit` (`toolkit_root` in `config/orchestration.yaml`) is on a clean **`ai`** branch with the spectral-only manifest. The launch preflight asserts this and fails cheaply with remediation if not (a dirty tree usually means a prior terminal authoring failure left edits to inspect). A reachable `human_handler`/channel is recommended so the lead/author can escalate; an honest `failed`/`needs_human` outcome is a *correct* result — never fabricated.

In [ ]:
authoring_request = ResearchRequest(
    question="""\
IXPE project (P1) — Intra-Observation Polarization Angle and Degree Stability in Bright BH-XRBs.

Targets and datasets:
- Cyg X-1: 01002901, 03002599
- Swift J1727.8-1613: 03005801, 03006001
- 4U 1630-47: 01250401
- LMC X-1: 02001901

This project quantifies the intra-observation stability of both polarization angle (PA) and polarization degree (PD) in long-exposure BH-XRB observations using time-resolved PCUBE analysis. For each observation, we divide the exposure into orbital-timescale segments (~5 ks), measure PA and PD in each segment, and apply chi-squared constancy tests to identify statistically significant variability in either quantity. Because PD is positively biased in the low-significance regime, the model-independent normalized Stokes parameters (Q/I, U/I) provide the statistically robust joint test, while the direct PA and PD constancy tests give the physically interpretable per-quantity assessment. Novel publishable results include: (1) a systematic catalog of PA and PD stability metrics for 6 high-exposure BH-XRB observations spanning 110-563 ks, (2) identification of any observations showing significant PA wander or PD variability beyond statistical noise, and (3) upper limits on PA and PD variability amplitude for sources consistent with constant polarization, constraining the geometric and physical stability of the polarization-producing regions.

Analysis prescription:
STEP 1 - DATA RETRIEVAL: Verify availability of Level-2 event files for all 6 observations: Cyg X-1 (01002901, 03002599), Swift J1727.8-1613 (03005801, 03006001), 4U 1630-47 (01250401), LMC X-1 (02001901). Confirm presence of ixpe_*_det1_evt2.fits, ixpe_*_det2_evt2.fits, ixpe_*_det3_evt2.fits for each obs_id. These observations range from 110 to 563 ks exposure, providing sufficient statistics for time-resolved polarimetry.

STEP 2 - SOURCE REGION DEFINITION AND EVENT EXTRACTION: For each observation:
  (a) Use xselect to define a circular source region of 60 arcsec radius centered on source coordinates.
  (b) Merge DU1, DU2, DU3 event files using xselect into a single combined event list.
  (c) Filter events to 2-8 keV energy range using xselect.
  Output: One filtered, merged event file per observation.

STEP 3 - TIME-RESOLVED POLARIZATION ANALYSIS: For each observation:
  (a) Determine total Good Time Interval (GTI) exposure and divide into N segments of approximately 5 ks each.
  (b) For each time segment, run xpbin with algorithm=PCUBE to generate a polarization cube using NEFF weighting.
  (c) Extract from each PCUBE: PD, PD_err, PA, PA_err, MDP99, and normalized Stokes parameters Q/I, U/I with errors.
  (d) Record segment start time (MJD), stop time, and effective exposure.
  Output per observation: N PCUBE files and a time-series table with columns [segment_id, t_start_mjd, t_stop_mjd, exposure_s, PD_pct, PD_err, PA_deg, PA_err, QI, QI_err, UI, UI_err, MDP99].

STEP 4 - OBSERVATION-INTEGRATED POLARIZATION: For each observation:
  (a) Run xpbin with algorithm=PCUBE on the full merged event file to obtain observation-averaged PD, PA, Q/I, U/I with errors.
  (b) Compute MDP99 for the full observation.
  (c) Record detection significance as PD/sigma_PD.
  Output: Integrated polarization parameters for each observation.

STEP 5 - PA AND PD CONSTANCY TESTS: For each observation:
  (a) Compile the time-series of PA and PD measurements from Step 3.
  (b) PA constancy: perform weighted chi-squared test chi2_PA = sum[(PA_i - PA_mean)^2 / PA_err_i^2], where PA_mean is the weighted mean PA.
  (c) PD constancy: perform weighted chi-squared test chi2_PD = sum[(PD_i - PD_mean)^2 / PD_err_i^2], where PD_mean is the weighted mean PD.
  (d) Compute degrees of freedom dof = N_segments - 1 for each test.
  (e) Calculate p-values from chi2 and dof using scipy.stats.chi2.sf for both PA and PD.
  (f) Flag observation as showing significant PA variability if p_value_PA < 0.01, and significant PD variability if p_value_PD < 0.01 (99% confidence).
  (g) CAVEAT: PD is a positively-biased quantity in the low-significance regime (PD/PD_err < ~3) and its distribution departs from Gaussian, so the chi2_PD test is only approximate when segment significances are low. Interpret chi2_PD alongside the model-independent Q/I, U/I constancy test of Step 6, which is unbiased and is the statistically robust joint reference.
  Output per observation: chi2_PA, chi2_PD, dof, p_value_PA, p_value_PD, pa_variability_flag, pd_variability_flag.

STEP 6 - STOKES PARAMETER CONSTANCY TEST (UNBIASED JOINT REFERENCE): For each observation:
  (a) Perform weighted chi-squared test for constancy of Q/I time series: chi2_QI = sum[(QI_i - QI_mean)^2 / QI_err_i^2].
  (b) Perform weighted chi-squared test for constancy of U/I time series: chi2_UI = sum[(UI_i - UI_mean)^2 / UI_err_i^2].
  (c) Compute combined 2D chi-squared: chi2_2D = chi2_QI + chi2_UI with dof_2D = 2*(N_segments - 1).
  (d) Calculate p-values for each test. Because Q/I and U/I are approximately Gaussian and unbiased, the 2D test is the primary, model-independent verdict on whether the polarization vector (PA and PD jointly) is constant.
  Output per observation: chi2_QI, chi2_UI, chi2_2D, p_value_QI, p_value_UI, p_value_2D.

STEP 7 - PA AND PD VARIABILITY AMPLITUDE QUANTIFICATION: For each observation:
  (a) Compute standard deviation of measured PA values (sigma_PA_measured) and of measured PD values (sigma_PD_measured).
  (b) Compute expected statistical scatter: sigma_PA_expected = sqrt(mean(PA_err_i^2)) and sigma_PD_expected = sqrt(mean(PD_err_i^2)).
  (c) If sigma_measured > sigma_expected for a quantity, compute intrinsic scatter sigma_intrinsic = sqrt(sigma_measured^2 - sigma_expected^2), separately for PA and PD.
  (d) Otherwise, report sigma_intrinsic as an upper limit (sigma_intrinsic < sigma_expected), separately for PA and PD.
  (e) Compute peak-to-peak ranges: PA_range = PA_max - PA_min and PD_range = PD_max - PD_min.
  Output: sigma_PA_intrinsic (or upper limit), sigma_PD_intrinsic (or upper limit), PA_range, PD_range.

STEP 8 - STRUCTURE FUNCTION ANALYSIS: For each observation with N >= 10 segments:
  (a) Compute first-order structure function of PA: SF_PA(tau) = <(PA(t+tau) - PA(t))^2>, and of PD: SF_PD(tau) = <(PD(t+tau) - PD(t))^2>.
  (b) Compute each SF at lag tau = 5 ks, 10 ks, 20 ks (if exposure permits).
  (c) Test for any trend in SF with lag that would indicate correlated PA or PD wandering.
  Output: PA and PD structure function values at each lag.

STEP 9 - CROSS-SOURCE COMPARISON: Compile results from all 6 observations into summary table.
  (a) Rank observations by PA constancy p-value and, separately, by PD constancy p-value to identify the most/least stable sources in each quantity.
  (b) Compute correlation between integrated MDP99 and chi2_PA/dof, and between MDP99 and chi2_PD/dof, using Spearman rank to check whether apparent variability is driven by measurement precision.
  (c) Compare mean intrinsic PA scatter and mean intrinsic PD scatter (or upper limits) between sources.
  (d) Identify whether any source shows variability in one quantity but not the other (e.g., stable PA with variable PD, or vice versa), which would help distinguish purely geometric reorientation from changes in scattering optical depth / emission-region physics.

STEP 10 - VISUALIZATION: Generate diagnostic and publication-quality figures using matplotlib:
  (a) Time-series plots for each observation with stacked panels sharing a common time axis: PA vs time and PD vs time, with error bars, overplotting the weighted mean as a horizontal line in each panel.
  (b) Q/I vs U/I Stokes plane plots for each observation showing all time segments as points with error ellipses, color-coded by time (this jointly visualizes PA and PD evolution).
  (c) Histograms of reduced chi-squared values (chi2_PA/dof and chi2_PD/dof) across all observations.
  (d) Summary figures: PA scatter vs observation duration and PD scatter vs observation duration for all sources.
  (e) Structure function plots SF_PA(tau) and SF_PD(tau) for observations with sufficient segments.

STEP 11 - OUTPUT PRODUCTS:
  (a) Master results table: pa_pd_stability_catalog.csv with columns [source, obs_id, exposure_ks, n_segments, PD_integrated, PD_integrated_err, PA_integrated, PA_integrated_err, chi2_PA, p_value_PA, chi2_PD, p_value_PD, dof, chi2_2D, p_value_2D, sigma_PA_intrinsic, sigma_PD_intrinsic, PA_range_deg, PD_range_pct, pa_variability_flag, pd_variability_flag].
  (b) Time-series data files per observation: {obs_id}_polarization_timeseries.csv with per-segment PA and PD measurements.
  (c) Summary statistics: pa_pd_variability_summary.txt reporting which observations pass/fail the PA and PD constancy tests at 99% confidence.
  (d) Publication-quality figures: pa_pd_timeseries_all.pdf, stokes_plane_evolution.pdf, chi2_histograms.pdf, structure_function.pdf.
""",
    context="""\
TEST SESSION — this exercises lead-driven tool authoring; evaluate the loop, not publication quality.
- All listed observations are IXPE observations (mission = "ixpe").
- Data is local-first.
- When a worker finds a capability missing, it must raise a tool_request (never fabricate numbers); the lead browses research_toolkit/lib read-only, decides extend-vs-new, and authors the missing tool(s) in dependency order (acquire -> reduce -> pcube). Each authored tool is gated and GPT co-lead reviewed before an atomic commit on branch 'ai'; the workers then rerun on it.
- Evaluation: verify the authoring loop runs end-to-end and the analysis completes on a freshly authored tool. An honest failed/needs_human outcome (e.g. a tool that cannot be made mergeable within the rework caps) is an acceptable, correct result — do NOT fabricate a polarization measurement. If the environment is broken, abort and report.
""",
)

AUTHORING_LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/P2/step-7")
AUTHORING_LOG_DIR.mkdir(parents=True, exist_ok=True)
authoring_jsonl = AUTHORING_LOG_DIR / f"p1_pa_pd_stability_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.jsonl"
authoring_monitor = RunMonitor(chatter=7, jsonl_path=authoring_jsonl, stream=sys.stdout)
# human_channel='cli' lets the lead/author escalate to you; use 'whatsapp'/'slack' for a remote run.
authoring_run = await run_research(authoring_request, monitor=authoring_monitor, human_channel="slack")
authoring_run.monitor.write_markdown(authoring_jsonl.with_suffix(".md"))
authoring_jsonl, authoring_jsonl.with_suffix(".md"), authoring_run.verdict


⏳ Launch preflight: a confirmation ping was sent to your human channel — reply to it to start the run (waiting up to 60s).
[00:44:29] #0001 run L1 run.start: Research run started | run_id=813039e538de
[00:44:29] #0002 librarian L3 librarian.literature.start: Literature node started
[00:44:42] #0007 librarian L5 librarian.search.results librarian_io: Librarian IO returned 16 raw search results | results=16
[00:44:43] #0008 librarian L5 librarian.consensus.start: Librarian IO abstraction debate started


/grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed_output', input_value=MemoryRecord(paper_id=Non...cu et al. 2005 MNRAS)."), input_type=MemoryRecord])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...u et al. 2005 MNRAS).")), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...u et al. 2005 MNRAS).")), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...u et al.

[00:46:48] #0020 librarian L5 librarian.lead.adjudication librarian_io_lead: IO lead synthesized final memory record


/grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed_output', input_value=MemoryRecord(paper_id=Non...scu et al. 2005 MNRAS."), input_type=MemoryRecord])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...cu et al. 2005 MNRAS.")), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...cu et al. 2005 MNRAS.")), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...cu et al

[00:46:49] #0021 librarian L5 librarian.local_retrieval.results: Local retrieval returned 0 literature paths | paths=0
[00:46:49] #0022 reviewer L3 reviewer.literature_review.start reviewer: Reviewer literature review started
[00:53:19] #0052 reviewer L2 reviewer.literature_review.verdict reviewer: Reviewer returned literature verdict | accepted=False more_lit=True analysis_rework=True
[00:53:19] #0053 librarian L3 librarian.literature.start: Literature node started


...

[00:53:24] #0058 librarian L5 librarian.search.results librarian_io: Librarian IO returned 0 raw search results | results=0
[00:53:25] #0059 librarian L5 librarian.local_retrieval.results: Local retrieval returned 0 literature paths | paths=0
[00:53:25] #0060 reviewer L3 reviewer.literature_review.start reviewer: Reviewer literature review started
[01:06:16] #0095 reviewer L2 reviewer.literature_review.verdict reviewer: Reviewer returned literature verdict | accepted=False more_lit=True analysis_rework=True
[01:06:16] #0096 librarian L3 librarian.literature.start: Literature node started


/grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed_output', input_value=ReviewVerdict(accepted=Fa...d tool on branch 'ai'."), input_type=ReviewVerdict])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](... tool on branch 'ai'.")), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](... tool on branch 'ai'.")), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](... tool o

[06/18/26 20:08:34] ERROR    Failed to receive or enqueue a message: ClientConnectionResetError,    ]8;id=5118284;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/slack_sdk/socket_mode/aiohttp/__init__.py\__init__.py]8;;\:]8;id=5118285;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/slack_sdk/socket_mode/aiohttp/__init__.py#305\305]8;;\
                             Cannot write to closing transport (s_8786220412371)                                   

[01:08:34] #0097 run L1 run.error: Research run failed | error=arxiv.HTTPError('https://export.arxiv.org/api/query?search_query=Cyg+X-1+IXPE+polarization+hard+state&id_list=&sortBy=relevance&sortOrder=descending&start=0&ma~


HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=Cyg+X-1+IXPE+polarization+hard+state&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)

In [ ]:
display(Markdown(authoring_run.markdown()))

## Inspect An Existing Monitor

In [ ]:
# Inspect the most recent monitor log. Override `log_dir`/`existing_jsonl` to point elsewhere
# (e.g. .../logs/dev/step-7 for an authoring run).
log_dir = Path("/data/fzhou/auto_research/logs/dev/step-12")
candidates = sorted(log_dir.glob("*.jsonl")) if log_dir.exists() else []
existing_jsonl = candidates[-1] if candidates else None

if existing_jsonl:
    rows = [l for l in existing_jsonl.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"{len(rows)} monitor events in {existing_jsonl}")
    md = existing_jsonl.with_suffix(".md")
    if md.exists():
        display(Markdown(md.read_text(encoding="utf-8")))
else:
    print(f"No logs yet in {log_dir}")